# Macro

### Create Hexagon

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

# dataA1 = pd.read_csv('../ComputedData/Accident/DataA1_with_MRT_Youbike_Parkinglot.csv')
dataA2 = pd.read_csv('../ComputedData/Accident/DataA2_with_MRT_Youbike_Parkinglot.csv')

dataA2.head()

In [3]:
def create_hexagon(center_x, center_y, size):
    angles = np.linspace(0, 2 * np.pi, 7)
    return Polygon([
        (center_x + size * np.cos(angle), center_y + size * np.sin(angle))
        for angle in angles
    ])

In [4]:
gdf = gpd.GeoDataFrame(dataA2, geometry=gpd.points_from_xy(dataA2.經度, dataA2.緯度))

gdf = gdf[
    (gdf['經度'] >= 119.7) & (gdf['經度'] <= 122.1) &
    (gdf['緯度'] >= 21.8) & (gdf['緯度'] <= 25.4)
]

# Step 2: 計算範圍 (bounding box)
bounds = gdf.total_bounds  # (minx, miny, maxx, maxy)
minx, miny, maxx, maxy = bounds

# hexagon 大小 (degree)
# 台灣約395,144
hex_size = 0.01  # 度數，1 度 ≈ 111 公里

# 六邊形的寬度和高度
width = hex_size * 2
height = np.sqrt(3) * hex_size

# 計算橫向和縱向的hexagon間距
x_spacing = width * 3/4
y_spacing = height

In [5]:
hexagons = []

x = minx
while x < maxx + width:
    y = miny
    while y < maxy + height:
        hex_center = (x, y)
        hexagon = create_hexagon(*hex_center, hex_size)
        hexagons.append(hexagon)
        y += y_spacing
    x += x_spacing

# 每列交錯排列，蜂巢狀
hexagons_shifted = []
row = 0
x = minx
while x < maxx + width:
    y = miny + (y_spacing / 2 if row % 2 else 0) 
    while y < maxy + height:
        hex_center = (x, y)
        hexagon = create_hexagon(*hex_center, hex_size)
        hexagons_shifted.append(hexagon)
        y += y_spacing
    x += x_spacing
    row += 1

hex_grid = gpd.GeoDataFrame(geometry=hexagons_shifted, crs='EPSG:4326')
# 將事故點分配到 hexagon
joined = gpd.sjoin(gdf, hex_grid, how='left', predicate='within')
# 統計每個 hexagon 的事故數量
hex_grid['num_accidents'] = joined.groupby('index_right').size()
# 沒有事故的 hexagon 設為 0
hex_grid['num_accidents'] = hex_grid['num_accidents'].fillna(0).astype(int)

/var/folders/w2/_g9w5yys0f171q4qqm469z1h0000gn/T/ipykernel_72176/2240936307.py:29: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  joined = gpd.sjoin(gdf, hex_grid, how='left', predicate='within')


In [ ]:
ax = hex_grid.plot(edgecolor='k', facecolor='none', figsize=(10, 10))
gdf.plot(ax=ax, color='red', markersize=5)

### Morans I
使用 Incremental Spatial Autocorrelation (ISA)

In [6]:
from libpysal.weights import DistanceBand
from esda.moran import Moran

# 1. 抓 hexagon 中心座標
coords = np.array(list(zip(hex_grid.geometry.centroid.x, hex_grid.geometry.centroid.y)))

# 2. 設定 threshold (距離)
max_threshold = 30000  # 最大 10 公里 (單位: 公尺)
step = 5000            # 每 1 公里做一次
thresholds = np.arange(step, max_threshold + step, step)

moran_I_list = []

for thresh in thresholds:
    print(f'計算 threshold: {thresh} 公尺')

    # 3. 建立距離矩陣（空間權重矩陣）
    w = DistanceBand(coords, threshold=thresh/111000, binary=True, silence_warnings=True)  # 注意：經緯度要換成度（1度約111公里）

    # 4. row-standardized 權重
    w.transform = 'r'

    # 5. 計算 Moran's I，用 hexagon 的事故數量 num_accidents
    moran = Moran(hex_grid['num_accidents'], w)

    # 6. 存結果
    moran_I_list.append((thresh, moran.I, moran.z_norm, moran.p_norm))  # 距離, Moran's I, Z-score, P-value


/var/folders/w2/_g9w5yys0f171q4qqm469z1h0000gn/T/ipykernel_72176/2228394305.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coords = np.array(list(zip(hex_grid.geometry.centroid.x, hex_grid.geometry.centroid.y)))


計算 threshold: 5000 公尺
計算 threshold: 10000 公尺
計算 threshold: 15000 公尺
計算 threshold: 20000 公尺
計算 threshold: 25000 公尺
計算 threshold: 30000 公尺


In [ ]:
thresholds, moran_I, z_scores, p_values = zip(*moran_I_list)

plt.plot(thresholds, z_scores, marker='o')
plt.xlabel('Threshold Distance (meters)')
plt.ylabel('Moran\'s I Z-score')
plt.title('Incremental Spatial Autocorrelation (ISA)')
plt.grid(True)
plt.show()

### GI

In [8]:
from esda.getisord import G_Local

threshold_distance = 10000 / 111000

# 建立空間權重矩陣
# threshold:距離還是要轉成度因為資料是 WGS84 (EPSG:4326)
w_gi = DistanceBand(coords, threshold=10000/111000, binary=True, silence_warnings=True)
w_gi.transform = 'r'

# 執行 Getis-Ord G*
gi = G_Local(hex_grid['num_accidents'], w_gi)

# 把 Z-score 和 P-value 加到 hex_grid
hex_grid['GiZScore'] = gi.Zs
hex_grid['GiPValue'] = gi.p_norm

In [ ]:
from shapely.geometry import box

taiwan = gpd.read_file('/Users/wangqiqian/Downloads/直轄市、縣(市)界線1140318/COUNTY_MOI_1140318.shp')

bbox = box(120, 21.53, 121.59, 25.18)  # (min_lon, min_lat, max_lon, max_lat)

# 篩選台灣行政區，geometry 有跟 bbox 相交的
taiwan = taiwan[taiwan.intersects(bbox)].copy()
taiwan = taiwan.to_crs(epsg=4326)

print(taiwan.crs)      # EPSG:4326
print(hex_grid.crs)    # EPSG:4326

hex_grid['hotspot'] = 'Not Significant'
hex_grid.loc[hex_grid['GiZScore'] > 1.96, 'hotspot'] = 'Hotspot' 
hex_grid.loc[hex_grid['GiZScore'] < -1.96, 'hotspot'] = 'Coldspot'

color_dict = {
    'Hotspot': 'red',
    'Coldspot': 'blue',
    'Not Significant': 'lightgrey'
}

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
taiwan.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)
hex_grid.plot(ax=ax, color=hex_grid['hotspot'].map(color_dict), edgecolor='k', linewidth=0.3, alpha=0.7)

plt.title('Getis-Ord Gi* Hotspot Analysis (Distance = 30km)')
plt.axis('off')
plt.show()